## Example 1: encrypt and decrypt with a raw AES-256 wrapping key

In [ ]:
import os

import aws_encryption_sdk
from aws_encryption_sdk import CommitmentPolicy
from aws_encryption_sdk.identifiers import EncryptionKeyType, WrappingAlgorithm
from aws_encryption_sdk.internal.crypto.wrapping_keys import WrappingKey
from aws_encryption_sdk.key_providers.raw import RawMasterKey


def raw_aes_key(key_id: bytes, material: bytes) -> RawMasterKey:
    """A local AES-256-GCM wrapping key: no KMS and no network, so the cell runs offline."""
    return RawMasterKey(
        provider_id="example-provider",
        key_id=key_id,
        wrapping_key=WrappingKey(
            wrapping_algorithm=WrappingAlgorithm.AES_256_GCM_IV12_TAG16_NO_PADDING,
            wrapping_key=material,
            wrapping_key_type=EncryptionKeyType.SYMMETRIC,
        ),
    )


client = aws_encryption_sdk.EncryptionSDKClient(commitment_policy=CommitmentPolicy.REQUIRE_ENCRYPT_REQUIRE_DECRYPT)
wrapping_key = raw_aes_key(b"key-1", os.urandom(32))
plaintext = b"attack at dawn " * 20
context = {"purpose": "notebook-test", "tenant": "example"}

ciphertext, encrypt_header = client.encrypt(source=plaintext, key_provider=wrapping_key, encryption_context=context)
assert plaintext[:15] not in ciphertext, "plaintext leaked into the ciphertext"
assert encrypt_header.algorithm.is_committing(), encrypt_header.algorithm

decrypted, decrypt_header = client.decrypt(source=ciphertext, key_provider=wrapping_key)
assert decrypted == plaintext
# The SDK may add its own entries (the signing suite stores a public key); ours must all round-trip.
assert context.items() <= decrypt_header.encryption_context.items(), decrypt_header.encryption_context
print(f"✓ aws_encryption_sdk {aws_encryption_sdk.__version__}: {len(plaintext)} B -> {len(ciphertext)} B "
      f"-> {len(decrypted)} B with {encrypt_header.algorithm.name}")

## Example 2: a tampered message does not decrypt

In [ ]:
import contextlib
import logging

from aws_encryption_sdk.exceptions import AWSEncryptionSDKClientError
from aws_encryption_sdk.identifiers import Algorithm
from cryptography.exceptions import InvalidSignature, InvalidTag


@contextlib.contextmanager
def quiet_sdk_logging():
    """The SDK logs a clean-up traceback ("Error on closing") after a failed decrypt; these fail on purpose."""
    sdk_logger = logging.getLogger("aws_encryption_sdk")
    previous_level = sdk_logger.level
    sdk_logger.setLevel(logging.CRITICAL)
    try:
        yield
    finally:
        sdk_logger.setLevel(previous_level)


# A non-signing committing suite, so the flipped byte is caught by the AES-GCM tag of its frame.
message, _ = client.encrypt(source=plaintext, key_provider=wrapping_key,
                            algorithm=Algorithm.AES_256_GCM_HKDF_SHA512_COMMIT_KEY)
tampered = bytearray(message)
tampered[len(tampered) // 2] ^= 0x01

try:
    with quiet_sdk_logging():
        client.decrypt(source=bytes(tampered), key_provider=wrapping_key)
except (AWSEncryptionSDKClientError, InvalidTag, InvalidSignature) as error:
    print(f"✓ tampered message rejected: {type(error).__name__}")
else:
    raise AssertionError("a message with a flipped byte decrypted without error")

## Example 3: the wrong wrapping key cannot decrypt

In [ ]:
from aws_encryption_sdk.exceptions import DecryptKeyError

# A key with a different id is never even tried: the SDK finds no matching encrypted data key.
try:
    with quiet_sdk_logging():
        client.decrypt(source=ciphertext, key_provider=raw_aes_key(b"key-2", os.urandom(32)))
except DecryptKeyError as error:
    print(f"✓ unknown key id rejected: {type(error).__name__}")
else:
    raise AssertionError("decrypted with a wrapping key whose id is not in the message")

# Same id, different key material: unwrapping the data key fails authentication.
try:
    with quiet_sdk_logging():
        client.decrypt(source=ciphertext, key_provider=raw_aes_key(b"key-1", os.urandom(32)))
except (AWSEncryptionSDKClientError, InvalidTag) as error:
    print(f"✓ wrong key material rejected: {type(error).__name__}")
else:
    raise AssertionError("decrypted with the wrong key material")

## Example 4: streaming encryption with small frames

In [ ]:
import io

payload = os.urandom(10_000)
with client.stream(mode="e", source=io.BytesIO(payload), key_provider=wrapping_key,
                   algorithm=Algorithm.AES_256_GCM_HKDF_SHA512_COMMIT_KEY, frame_length=1024) as encryptor:
    stream_ciphertext = encryptor.read()
    assert encryptor.header.frame_length == 1024, encryptor.header.frame_length

with client.stream(mode="d", source=io.BytesIO(stream_ciphertext), key_provider=wrapping_key) as decryptor:
    chunks = []
    while True:
        chunk = decryptor.read(3000)  # read in pieces that do not line up with the frames
        if not chunk:
            break
        chunks.append(chunk)

assert b"".join(chunks) == payload
print(f"✓ streamed {len(payload):,} B through {-(-len(payload) // 1024)} frames of 1024 B and back "
      f"in {len(chunks)} reads")

## Example 5: the commitment policy refuses non-committing messages

In [ ]:
from aws_encryption_sdk.exceptions import ActionNotAllowedError

# A client that may still write the older, non-committing algorithm suites.
legacy_client = aws_encryption_sdk.EncryptionSDKClient(commitment_policy=CommitmentPolicy.FORBID_ENCRYPT_ALLOW_DECRYPT)
legacy_message, legacy_header = legacy_client.encrypt(source=b"legacy", key_provider=wrapping_key)
assert not legacy_header.algorithm.is_committing(), legacy_header.algorithm
assert legacy_client.decrypt(source=legacy_message, key_provider=wrapping_key)[0] == b"legacy"

# The strict client from Example 1 must refuse it.
try:
    with quiet_sdk_logging():
        client.decrypt(source=legacy_message, key_provider=wrapping_key)
except ActionNotAllowedError as error:
    print(f"✓ {legacy_header.algorithm.name} refused under REQUIRE_ENCRYPT_REQUIRE_DECRYPT: {error}")
else:
    raise AssertionError("the REQUIRE_ENCRYPT_REQUIRE_DECRYPT client decrypted a non-committing message")